## Install

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 8.2 MB/s eta 0:00:0000:0100:01


## Data

In [1]:
# import json
import pandas as pd
import numpy as np
# from collections import Counter
# import matplotlib.pyplot as plt
# import seaborn as sns

In [2]:
df = pd.read_csv("data/data_latest.csv")

df

,review_id,date,review_text,topic,subtopic,sentiment
0,1000087,2025-09-19,Вклад «Новые деньги» невозможно оформить без п...,Вклады,NaN,Negative
1,999494,2025-09-18,В июне 2025 года я порекомендовал премиальную ...,Дебетовые карты,NaN,Negative
2,999142,2025-09-17,Мошенниччиские аперации в интересах Ренессанс ...,Обслуживание,NaN,Negative
3,998360,2025-09-15,Купил услугу Газпром Бонус «Премиум» за 2 990 ...,Дебетовые карты,NaN,Negative
4,998516,2025-09-15,Производил оформление открытия срочного банков...,Вклады,«Накопительный»,Negative
...,...,...,...,...,...,...
4773,7470,2011-04-07,Ужастное обслуживание! Мало того потеряли доку...,Обслуживание,NaN,Negative
4774,7049,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,Negative
4775,5221,2011-01-25,"Мало того уже прошла неделя, а ПТС так и не ве...",Автокредиты,NaN,Negative
4776,5053,2011-01-16,Газпромбанк– отличный банк с отличными сотрудн...,Ипотека,NaN,Positive


In [3]:
df["topic"].value_counts()

topic
Дебетовые карты               1147
Обслуживание                   701
Кредиты наличными              610
Кредитные карты                608
Вклады                         469
Дистанционное обслуживание     365
Другие услуги                  365
Ипотека                        225
Автокредиты                    113
Рефинансирование кредитов       84
Рефинансирование ипотеки        27
Обмен валют                     26
Мобильное приложение            18
РКО                             11
Денежные переводы                4
Условия                          3
Эквайринг                        1
Кредитование бизнеса             1
Name: count, dtype: int64

## Synthetic data

### prompts

```

Твоя задача - суммаризировать и классифицировать отзывы клиентов Газпромбанка (ГПБ), тебе нужно определить о чём клиенты говорят в этих отзывах (обычно это продукты или услуги) и как они отзываются об этих продуктах.
Для каждого отзыва тебе нужно определить о каких темах говорит клиент. Примером таких тем может быть: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Помимо этих могут возникать и другие.
Тебе также необходимо определить тональность клиента, по отношению к каждому определённому продукту, хорошо или плохо они отзываются о продукте. Тональность может быть одной из 3: positive, negative, neutral. ВАЖНО! Neutral указывается тогда, когда тема упомянута, но к ней нет никаких плюсов / минусов, это просто факт или упоминание, не забывай про это, иначе можно ничего не опрделить к нейтральным отзывам, а их много.
В отзывах часто можно встретить ****. Это либо конфиденциальные данные, такие как номера телефона, или, что более вероятно, маты.
Если ты не можешь определить темы, исходя из отзыва, возвращай пустой массив тем и тональностей.

Для каждого отзыва ответь в виде json:
{
    [
        {
            "summarized_review": "краткий текст первого отзыва",
            "topics": ["продукт1", "продукт2"],
            "sentiments" : ["positive", "negative"]
        },
        {
            "summarized_review": "краткий текст второго отзыва",
            "topics": ["продукт1", "продукт2", "продукт3"],
            "sentiments" : ["neutral", "neutral", "neutral"]
        }
    ]
}

Вот конкретный пример:
Отзыв 1: Заказав две дебетовые карты банка, пожалела, что связалась с этим банком. Карты получить не смогла, о чем предварительно (за день до доставки карт) позвонила на горячую линию и сообщила. Поддержка по телефону ответила спасибо, но предупредила, что скорее всего сотрулник не получит информацию об отказе. Очень интересно. Затем начались смс и звонки. От сотрудника доставки 3 звонка и несколько смс. И с этого момента, ежедневно, поступают звонки от Газпромбанка с формулировкой: Здравствуйте. Вы. ФИО. С учётом телефонного мошенничества, я не могу сообщить им запрошенные данные о чем и сообщаю. В ответ или: тогла мы перезвоним вам позже или опять повторяют запрос. В воскресенье поступило 5 таких звонков с разных номеров. Далее ещё 3 на следующий день. Перезвонив в банк, уточняю у сотрудника, являются ли все номера-номерами банка. Ответ — да. Оставляю заявление о запрете звонков и смс от банка. Но, запрет начинает работать через 72 часа после заявления. И тут же ещё 3 звонка. Объяснять что либо-бесполезно. Ругаться- тоже. Сотрудник звонит и просто смеётся в трубку. На каком основании сотрудники банка так себя ведут? Мешают работать, не дают отдыхать, нарушают мои права, используя отсутствие возможности даже подать жалобу, так как при звонке они называют лишь имя и не дают информацию о себе никакую (но мои данные они запрашивают). Далее поступают звонки в 9 утра 2, днем 2. Говорю сотруднику, что не имею и не хочу иметь никаких продуктов банка, прерывают и: так как Вы не дали ответ на вопрос, то по регламенту перезвоним Вам позже. Как итог- могу лишь передать огромное спасибо владельцам банка за такое замечательное отношение. Впредь больше ни в коем случае не буду рекомендовать банк своим родным. Хотя раньше считала, что банк соблюдает все законы как государства, так и этики.

Отзыв 2: Видно, что над приложением поработали. Учли наверное опыт по их Телекарду. Прилага не виснет, проплаты проходят почти что мгновенно, много чего можно сделать через нее: начиная от заявки на кредит, заканчивая открытием накопительного счета. Я знаю, что любые приложения виснут, однако программу от ГПБ могу назвать относительно стабильной. Бывают конечно моменты, но от этого никто не застрахован. Тут главное снизить количество этих моментов. И разработчикам удалось!

Твой ответ должен выглядеть так:
{
    [
        {
            "summarized_review": "Клиент не мог вовремя забрать заказанную дебетовую карту и в поддержке не смогли помочь с этим вопросом. Клиент жалуется на спам звонки от сотрудников банка и выражает недоверие им.",
            "topics": ["Дебетовая карта", "Дистанционное обслуживание"],
            "sentiments": ["neutral", "negative"]
        },
        {
            "summarized_review": "Клиент положительно оценивает улучшения в мобильном приложении: стабильная работа, быстрые платежи, много функций для кредитов и счетов.",
            "topics": ["Мобильное приложение", "Кредиты", "Вклады"],
            "sentiments": ["positive", "neutral", "neutral"]
        }
    ]
}


Вот отзывы клиентов, которые необходимо обработать:

Отзыв 1: Банк не снимает арест с моего счета, хотя судебный пристав уже дважды отправлял постановление о снятии ареста с моих денежных средств!

Отзыв 2: Подала заявку на цифровую кредитную карту 21.06.2021Г. Через приложение Телекард. Через два дня, путем переговоров и отправкой соответствующих документов, одобрен был лимит 27 000 руб. Это было 23.06.2021. Сказали ждать готовность карты в следующем смс. Никакой смс соответственно не пришло в течении 2-х недель. Зашла в приложение, а так показана эта карта, заблокирована банком с лимитом 0 руб. Я позвонила на горячую линию, чтобы узнать, когда все-таки будет готова карта. Она же цифровая. , А не пластик. На что консультант ответил, что карта должна была быть готова сразу, но сотрудники пропустили все сроки и он подал заново заявку на рассмотрение специалистов, чтобы разобрались с этой проблемой. По сей день тишина. Что за отношение такое к клиентам непонятно.

Отзыв 3: Не советую брать сим-карты гпб-мобайл. Я имела неосторожность взять, прельстившись выгодными условиями, в итоге мне выдали одну сим-карту, а номер дали другой, и я пополнила счёт на сим-карте, которая мне не принадлежала. Когда всё вскрылось, в банке поохали-поахали, но деньги вернуть отказались, даже когда я написала заявление.

Отзыв 4: Доброго дня. По эл. Почте пришло сообщение выгодное предложение до 13 декабря. Решил рефенансировать если 1500000 руб. Берёшь 5,5%. Но это обман. Когда позвонил мне сказали что 6,9%. Я являюсь зарплатным клиентом. Мой кредитный рейтинг 925. Для кого эти предложения? Когда написал на горячую линию мне ответили что это от 5,5% (как в рекламе от 20 лет) хотя в рекламе нет ОТ. Получается заведомо обманываете клиентов, ещё и по электронке предлогаете услуги которые не предоставляете. Разочарован вашим банком.

Отзыв 5: Пользовался картой, потом просто лежала. Пришел долг через год. Погасил его и карту заблокировал. Через какое-то время опять повесили долг. Поддержка не реагирует вообще никак. Какой-то робот делает обычную отписку.

```

```
Вот ещё 5 отзывов:

Отзыв 6: 20.08.2025 Оставил заявку на выдачу карты Газпром Бонус Премиум. Назначили встречу с представителем банка на 26.08.2025 Г. (Встретится в отделении возможности в СПб нет, что является минусом)
Представитель отзвонился 26.08.25 В 13:15 соообщить о том, что сегодня встреча не состоится, а колл центр мне позвонит и назначит новую встречу.
Итог. Встреча не состоялась, карты нет, новое время для встречи не назначено, планы на день приедется менять. Премиальное обслуживание от Газпромбанка.

Отзыв 7: Видно, что над приложением поработали. Учли наверное опыт по их Телекарду. Прилага не виснет, проплаты проходят почти что мгновенно, много чего можно сделать через нее: начиная от заявки на кредит, заканчивая открытием накопительного счета. Я знаю, что любые приложения виснут, однако программу от ГПБ могу назвать относительно стабильной. Бывают конечно моменты, но от этого никто не застрахован. Тут главное снизить количество этих моментов.
И разработчикам удалось!

Отзыв 8: Сбелала заявку на сайте, карту привезли, условия не объяснили, информации нет. Позвонила, пришлось «висеть на трубке» 10 мин. Переключали 5 раз друг на друга, сотрудники знают только одну фразу «я не могу отвечать за другого сотрудника». Такого ужасного мобильного приложения и отношения к клиентам я нигде не встречала. Это просто «дно»! Подумайте 10 раз, прежде чем туда обратиться. В банкоматах 5000 купюру не принимают, ездила в 3 места, везде так. Деньги забираю и больше ни ногой!

Отзыв 9: Пишу повторный отзыв. Прошел месяц. Обратился в банк 04.11.2024 Зарегистрировали обращение под номером № 2039528 от 04.11.2024. Суть проблемы следующая: прошёл процедуру банкротства, о чем есть определение арбитражного суда от 27.06.2024.
Банк не убрал информацию о кредите, который был списан. Этот кредит не даёт мне заблокировать счёт и карту. Приложение даёт ошибку что-то вроде «Невозможно закрыть счёт так как с него происходит списание за кредит»
По обращению был поставлен срок рассмотрения до 19.11.2024. Настало 20 ноября, начал обращаться в банк. Сказали, что срок рассмотрения 15 рабочих дней. Я согласился. Прошло 15 рабочих дней, мне сказали, что в определенных случаях срок рассмотрения обращения может быть увеличен на 10 дней. Тут я тоже согласился.
Но вот сегодня уже 36 рабочий день)
На все мои звонки и вопросы специалисты банка «Сожалеют», что так произошло, говорят, что моё обращение в работе и мямлят что-то не совсем внятное.
Видимо эти специалисты плевать хотели на клиентов и на Федеральное законодательство. Есть ведь строго определенные сроки, но реагировать на это они не хотят.

Отзыв 10: Банк для дела. Всё грамотно и понятно. Достаточно доступно. Процент низкий. Расчёты доступны. Досрочное погашение.

```

Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Составить краткую текстовую сводку (summarized_review).
2.  Определить все продукты/услуги (темы), о которых упоминает клиент.
3.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного упоминания продукта или услуги, не включай его.
-   **Символы `****`:** Это либо конфиденциальные данные (номера телефонов), либо ненормативная лексика. Учитывай общий контекст вокруг них для определения тональности.
-   **Если темы нет:** Если в отзыве невозможно определить ни одну тему, верни пустой массив `topic_sentiment_pairs`.

Вот список всех продуктов банка на сегодняшний день:
* **дебетовые карты:**
    * Умная дебетовая карта «Мир»
    * Премиальная карта Mir Supreme
    * Дебетовая карта с кэшбэком для самозанятых
    * Умная дебетовая карта «Мир»
    * Карта для автолюбителей «Газпромбанк—Газпромнефть»
    * Виртуальная дебетовая карта ГПБ&ФК «Зенит»
    * Дебетовая Пенсионная карта
* **Кредитные карты:**
    * Кредитная карта с льготным периодом до 120 дней
    * Простая кредитная карта
    * Кредитная карта 90 дней
    * Кредитная карта 180 дней Премиум
    * Кредитная карта для самозанятых
* **Накопительные счета:**
    * Накопительный счет
    * «Ежедневная выгода»
    * «Ежедневный процент»
    * «Премиум»
    * Социальный счет
* **Вклады:**
    * Вклад «Новые деньги»
    * Вклад «Ключевой момент»
    * Вклад «Копить»
    * Вклад «В Плюсе»
    * Вклад «Расширяй возможности»
    * Социальный вклад
* **Кредиты:**
    * Кредит наличными
    * Кредит наличными под залог недвижимости
    * Кредит на авто и другие цели
    * Рефинансирование потребительских кредитов
    * Дачный кредит
    * Кредит на образование
    * Кредит наличными для бюджетников
* **Другие услуги банка:**
    * Газпромбанк Мобайл
    * Газпромбанк Travel
    * Gazprom Pay
    * GorodPay
    * Газпром Бонус
    * Страховые и сервисные продукты
    * Депозитарные услуги


**Формат ответа — строго JSON:**
[
    {
        "id" : "исходный_id_отзыва",
        "summarized_review": "Краткая сводка отзыва на русском языке.",
        "topic_sentiment_pairs": [
            {
                "topic": "Название темы 1",
                "sentiment": "positive/negative/neutral"
            },
            {
                "topic": "Название темы 2",
                "sentiment": "positive/negative/neutral"
            }
        ]
    }
]


**Пример:**
Отзыв 1: Мобильное приложение просто ужасное, постоянно вылетает. А вот вклад оформил недавно — условия хорошие.


Отзыв 2: Звонил узнать насчет ипотеки, но в поддержке ничем не помогли.

Ответ:
[
    {
        "id" : 1,
        "summarized_review": "Клиент критикует мобильное приложение за нестабильность, но хвалит условия по вкладу.",
        "topic_sentiment_pairs": [
            {
                "topic": "Мобильное приложение",
                "sentiment": "negative"
            },
            {
                "topic": "Вклады",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id" : 2,
        "summarized_review": "Поддержка не смогла помочь клиенту по вопросу с ипотекой.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Ипотека",
                "sentiment": "neutral"
            }
        ]
    },
]

**Вот отзывы клиентов, которые необходимо обработать:**
{}

Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Определить все продукты/услуги (темы), о которых упоминает клиент.
2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного упоминания продукта или услуги, не включай его.
-   **Символы `****`:** Это либо конфиденциальные данные (номера телефонов), либо ненормативная лексика. Учитывай общий контекст вокруг них для определения тональности.
-   **Если темы нет:** Если в отзыве невозможно определить ни одну тему, верни пустой массив `topic_sentiment_pairs`.

Вот список всех продуктов банка на сегодняшний день:
* **дебетовые карты:**
    * Умная дебетовая карта «Мир»
    * Премиальная карта Mir Supreme
    * Дебетовая карта с кэшбэком для самозанятых
    * Карта для автолюбителей «Газпромбанк—Газпромнефть»
    * Виртуальная дебетовая карта ГПБ&ФК «Зенит»
    * Дебетовая Пенсионная карта
* **Кредитные карты:**
    * Кредитная карта с льготным периодом до 120 дней
    * Простая кредитная карта
    * Кредитная карта 90 дней
    * Кредитная карта 180 дней Премиум
    * Кредитная карта для самозанятых
* **Накопительные счета:**
    * Накопительный счет
    * «Ежедневная выгода»
    * «Ежедневный процент»
    * «Премиум»
    * Социальный счет
* **Вклады:**
    * Вклад «Новые деньги»
    * Вклад «Ключевой момент»
    * Вклад «Копить»
    * Вклад «В Плюсе»
    * Вклад «Расширяй возможности»
    * Социальный вклад
* **Кредиты:**
    * Кредит наличными
    * Кредит наличными под залог недвижимости
    * Кредит на авто и другие цели
    * Рефинансирование потребительских кредитов
    * Дачный кредит
    * Кредит на образование
    * Кредит наличными для бюджетников
* **Другие услуги банка:**
    * Газпромбанк Мобайл
    * Газпромбанк Travel
    * Gazprom Pay
    * GorodPay
    * Газпром Бонус
    * Страховые и сервисные продукты
    * Депозитарные услуги


**Формат ответа — строго JSON:**
{
    "id" : "исходный_id_отзыва",
    "topic_sentiment_pairs": [
        {
            "topic": "Название темы 1",
            "sentiment": "positive/negative/neutral"
        },
        {
            "topic": "Название темы 2",
            "sentiment": "positive/negative/neutral"
        }
    ]
}


**Примеры:**
Отзыв 1: Мобильное приложение просто ужасное, постоянно вылетает. А вот вклад оформил недавно — условия хорошие.

Ответ:
{
    "id" : 1,
    "topic_sentiment_pairs": [
        {
            "topic": "Мобильное приложение",
            "sentiment": "negative"
        },
        {
            "topic": "Вклады",
            "sentiment": "positive"
        }
    ]
}


Отзыв 2: Звонил узнать насчет ипотеки, но в поддержке ничем не помогли.

Ответ:
{
    "id" : 2,
    "topic_sentiment_pairs": [
        {
            "topic": "Дистанционное обслуживание",
            "sentiment": "negative"
        },
        {
            "topic": "Ипотека",
            "sentiment": "neutral"
        }
    ]
},

**Вот отзывы клиентов, которые необходимо обработать:**
{}

**Ответ (только JSON, без дополнительного текста):**

### smt

In [11]:
df

,review_id,date,review_text,topic,subtopic,sentiment
0,1000087,2025-09-19,Вклад «Новые деньги» невозможно оформить без п...,Вклады,NaN,Negative
1,999494,2025-09-18,В июне 2025 года я порекомендовал премиальную ...,Дебетовые карты,NaN,Negative
2,999142,2025-09-17,Мошенниччиские аперации в интересах Ренессанс ...,Обслуживание,NaN,Negative
3,998360,2025-09-15,Купил услугу Газпром Бонус «Премиум» за 2 990 ...,Дебетовые карты,NaN,Negative
4,998516,2025-09-15,Производил оформление открытия срочного банков...,Вклады,«Накопительный»,Negative
...,...,...,...,...,...,...
4773,7470,2011-04-07,Ужастное обслуживание! Мало того потеряли доку...,Обслуживание,NaN,Negative
4774,7049,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,Negative
4775,5221,2011-01-25,"Мало того уже прошла неделя, а ПТС так и не ве...",Автокредиты,NaN,Negative
4776,5053,2011-01-16,Газпромбанк– отличный банк с отличными сотрудн...,Ипотека,NaN,Positive


In [25]:
test = [
    {
        "id": 196550,
        "summarized_review": "Клиент жалуется на некомпетентность сотрудника, который допустил ошибку в фамилии при выпуске дебетовой карты. Из-за этой ошибки клиент не смог получить карту и понес дополнительные расходы, а банк не смог оперативно решить проблему.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 920737,
        "summarized_review": "Клиент недоволен длинными очередями и медленной работой сотрудников. Основная претензия заключается в том, что банк не выплатил обещанный кэшбэк в 5000 рублей за выпуск карты UnionPay, сославшись на то, что акция была выдумана сотрудником.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 302437,
        "summarized_review": "Клиент жалуется на задержку снижения процентной ставки по ипотеке после регистрации договора залога, как было предусмотрено договором. Из-за бюрократических проволочек и некомпетентности сотрудников банка, клиенту пришлось заплатить по старой, более высокой ставке.",
        "topic_sentiment_pairs": [
            {
                "topic": "Ипотека",
                "sentiment": "negative"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 85814,
        "summarized_review": "Клиент получил карту по предложению банка, но не может ей пользоваться уже две недели из-за задержки с активацией, что превышает сроки, указанные в договоре. Техническая поддержка не может решить проблему.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредитные карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 460586,
        "summarized_review": "Новый клиент дважды пытался организовать встречу с представителем банка для получения услуг, но обе встречи были отменены без последующей связи. Попытка дозвониться в поддержку также не увенчалась успехом из-за долгого ожидания.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 743406,
        "summarized_review": "Клиент прождал в очереди в отделении банка почти два часа, при этом пропустив вперед клиентов, пришедших позже. Сотрудники не смогли объяснить причину задержки.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 978279,
        "summarized_review": "Клиент жалуется на неожиданное изменение условий по кредитной карте. Несмотря на своевременные платежи, банк начал начислять штрафы и пени, предположительно из-за одностороннего сокращения льготного периода со 180 до 90 дней без уведомления.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредитные карты",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 387604,
        "summarized_review": "Клиент, являясь зарплатным клиентом с высоким кредитным рейтингом, получил предложение о рефинансировании по ставке 5,5%, но при обращении в банк ему предложили ставку 6,9%. Клиент считает это обманом и недобросовестной рекламой.",
        "topic_sentiment_pairs": [
            {
                "topic": "Рефинансирование кредитов",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 488600,
        "summarized_review": "Клиент недоволен графиком работы кассиров в отделении, которые уходят на обед одновременно с обеденным перерывом у бюджетных служащих, создавая неудобства для клиентов.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 391327,
        "summarized_review": "Клиент благодарит сотрудницу банка Ольгу за помощь и поддержку на всех этапах оформления ипотеки.",
        "topic_sentiment_pairs": [
            {
                "topic": "Ипотека",
                "sentiment": "positive"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 785993,
        "summarized_review": "Клиент выполнил все условия акции по карте, но не получил обещанное вознаграждение в 5000 рублей в указанный срок. Также отмечаются проблемы с работой чата поддержки.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 299987,
        "summarized_review": "Клиент выражает восторг от профессионализма сотрудницы банка Анны, которая быстро и грамотно проконсультировала по кредиту и переводу денежных средств.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредиты наличными",
                "sentiment": "positive"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 593082,
        "summarized_review": "Клиент пришел в отделение для консультации по вкладу, но так и не дождался своей очереди, прождав полтора часа. Обслуживание оценено как отвратительное.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Вклады",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 777406,
        "summarized_review": "Постоянный клиент банка выражает полное удовлетворение услугами, от кредитов до обмена валют, и особенно отмечает высокое качество обслуживания и готовность сотрудников помочь.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредиты наличными",
                "sentiment": "positive"
            },
            {
                "topic": "Обмен валют",
                "sentiment": "positive"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 543154,
        "summarized_review": "Клиент в течение трех месяцев не мог получить от банка четкий ответ по поводу необходимости страхования титула по ипотеке. Из-за противоречивой и некомпетентной информации от сотрудников поддержки и офиса, банк в итоге повысил клиенту процентную ставку.",
        "topic_sentiment_pairs": [
            {
                "topic": "Ипотека",
                "sentiment": "negative"
            },
            {
                "topic": "Рефинансирование ипотеки",
                "sentiment": "neutral"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 526078,
        "summarized_review": "Клиент жалуется на невозможность снять или перевести деньги с дебетовой карты. Отмечаются высокие комиссии за перевод на другие карты и отсутствие функции перевода по номеру телефона как в приложении, так и в банкоматах.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Мобильное приложение",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 200145,
        "summarized_review": "Клиент недоволен устаревшими процедурами банка: активация онлайн-банкинга занимает 48 часов, а дополнительную карту можно активировать только в офисе, что создает неудобства.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 493546,
        "summarized_review": "Клиент жалуется на списание более 22 000 рублей со счета и требует вернуть деньги, не уточняя подробностей операции.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 400799,
        "summarized_review": "Клиент обвиняет банк в введении в заблуждение, так как низкие рекламные ставки по кредитам достигаются только при условии покупки дорогой страховки в самом банке, что в итоге значительно увеличивает реальную процентную ставку.",
        "topic_sentiment_pairs": [
            {
                "topic": "Потребительский кредит",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 670563,
        "summarized_review": "Клиент жалуется, что полученную карту практически нигде не принимают к оплате, что делает ее бесполезной, в том числе для поездок за границу.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 721466,
        "summarized_review": "Клиент столкнулся с неработающей картой и хамством оператора поддержки. После звонка карта оказалась заблокирована, якобы по звонку третьего лица. Клиент подозревает сотрудника в неправомерных действиях или утечке данных.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Мобильное приложение",
                "sentiment": "neutral"
            }
        ]
    },
    {
        "id": 756166,
        "summarized_review": "Клиент отказался от получения дебетовой карты из-за грубого и навязчивого поведения представителя банка, который пытался принудительно выдать еще и кредитную карту.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Кредитные карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 951266,
        "summarized_review": "Клиент описывает множество проблем с кредитной картой Union Pay: навязанная страховка, неработающая за границей карта, списание процентов в льготный период и, в конечном итоге, списание комиссии за обслуживание заблокированной карты.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредитные карты",
                "sentiment": "negative"
            },
            {
                "topic": "Мобильное приложение",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 999494,
        "summarized_review": "Клиент участвовал в акции \"Приведи друга\" и выполнил все условия. Однако вознаграждение получил только его друг, а клиент свою часть выплаты так и не дождался, получая в поддержке лишь стандартные отписки.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 979957,
        "summarized_review": "Банк без предупреждения заблокировал дебетовую карту клиента по 115-ФЗ. Клиент столкнулся с неинформативной поддержкой и требованием предоставить неопределенный пакет документов, а после отказа в разблокировке вынужден ждать до 30 дней возврата своих денег.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Мобильное приложение",
                "sentiment": "neutral"
            }
        ]
    },
    {
        "id": 398735,
        "summarized_review": "Клиентка потеряла две недели на оформление автокредита, прошла несколько встреч и подписала документы, но в итоге получила отказ. За это время автомобиль, который она планировала купить, был продан.",
        "topic_sentiment_pairs": [
            {
                "topic": "Автокредиты",
                "sentiment": "negative"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 670623,
        "summarized_review": "Клиент доволен дебетовой картой Union Pay. Карта успешно использовалась для оплаты в Китае и для снятия наличных с невысокой комиссией в Китае и Южной Корее.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 490225,
        "summarized_review": "Клиент благодарит сотрудницу Татьяну за быструю, качественную и вежливую помощь в оформлении документов по программе господдержки семей.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "positive"
            },
            {
                "topic": "Социальные программы/Господдержка",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 842738,
        "summarized_review": "Клиентка крайне недовольна кредитной картой, утверждая, что после окончания льготного периода банк начал списывать огромные проценты, которые значительно превышают ожидаемые расчеты. Поддержка не смогла объяснить причину таких начислений.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредитные карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 288352,
        "summarized_review": "Клиент недоволен необходимостью каждый раз тратить час в очереди в отделении для подачи заявления на досрочное погашение кредита и отсутствием возможности сделать это дистанционно.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Потребительский кредит",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Рефинансирование кредитов",
                "sentiment": "neutral"
            }
        ]
    },
    {
        "id": 369074,
        "summarized_review": "Клиент стал жертвой мошенников, которые представились службой безопасности банка и владели всеми его личными данными. После кражи денег банк очень долго рассматривал претензию и в итоге отказал в возврате средств.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредиты наличными",
                "sentiment": "neutral"
            },
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Безопасность",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 479558,
        "summarized_review": "Клиент закрыл вклад спустя полтора года и остался полностью доволен: процедуры открытия и закрытия были простыми, а проценты начислены корректно.",
        "topic_sentiment_pairs": [
            {
                "topic": "Вклады",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 687070,
        "summarized_review": "Клиент жалуется на ужасное обслуживание в отделении, включая долгое ожидание и некомпетентность сотрудников, а также на безрезультатность письменных обращений по вопросам обслуживания карт.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 457558,
        "summarized_review": "Клиент оформил платную подписку для получения скидки на ипотеку, но банк отказался ее предоставить, ссылаясь на противоречивые причины. Клиент считает подписку обманом.",
        "topic_sentiment_pairs": [
            {
                "topic": "Ипотека",
                "sentiment": "negative"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 364721,
        "summarized_review": "Зарплатный клиент взял кредит в банке, привлекшись более низкой процентной ставкой по сравнению с другими банками. Процесс одобрения и получения денег был быстрым и без проблем.",
        "topic_sentiment_pairs": [
            {
                "topic": "Потребительский кредит",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 408469,
        "summarized_review": "Клиент приехал в отделение банка за 200 км по предварительной записи, но не смог попасть внутрь из-за плохой организации входа.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 484765,
        "summarized_review": "Клиент столкнулся с тем, что начисленные проценты по вкладу были немедленно списаны банком в счет уплаты штрафа, из-за чего он решил забрать все деньги из банка.",
        "topic_sentiment_pairs": [
            {
                "topic": "Вклады",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 854420,
        "summarized_review": "Клиент не получил обещанный кэшбэк по акции, несмотря на выполнение всех условий. Также клиент недоволен бесполезной работой онлайн-чата и внезапным повышением порога для конвертации бонусов в рубли.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 396668,
        "summarized_review": "Клиент описывает ужасный опыт оформления ипотеки: некомпетентный менеджер и отсутствие туалета для клиентов в отделении, что создало проблемы для беременной женщины и ребенка.",
        "topic_sentiment_pairs": [
            {
                "topic": "Ипотека",
                "sentiment": "negative"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 787934,
        "summarized_review": "Банкомат принял у клиента крупную сумму денег (300 000 рублей), но не зачислил на счет. На горячей линии сообщили, что рассмотрение заявки может занять 30 и более дней, оставив клиента без денег на неопределенный срок.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 321730,
        "summarized_review": "Клиент жалуется на медленное обслуживание в отделении, где из трех окон работает только одно, а остальные сотрудники заняты личными делами и разговорами.",
        "topic_sentiment_pairs": [
            {
                "topic": "Офисное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 620431,
        "summarized_review": "Клиент столкнулся с мошеннической схемой привлечения: под предлогом трудоустройства его вынудили оформить карту Газпромбанка, после чего 'работодатель' исчезал. Банк на жалобу ответил формальной отпиской.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 803767,
        "summarized_review": "Клиент возмущен условиями по карте Union Pay, где за выпуск взимается плата в 5000 рублей, несмотря на заявленное бесплатное обслуживание.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 859826,
        "summarized_review": "Клиент жалуется на невыплату вознаграждения по акции 'Приведи друга'. Банк предоставляет противоречивую информацию, сначала отрицая участие в акции, а затем обвиняя друга в невыполнении условий, и уклоняется от выплаты.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 916202,
        "summarized_review": "Клиент выполнил все условия акции по возврату 5000 рублей за выпуск карты Union Pay, но банк отказал в выплате без объяснения причин. Дозвониться до поддержки невозможно.",
        "topic_sentiment_pairs": [
            {
                "topic": "Дебетовые карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 405193,
        "summarized_review": "Клиент остался очень доволен процессом рефинансирования кредита: заявку одобрили моментально, а сотрудница банка подробно все объяснила и помогла с установкой мобильного приложения.",
        "topic_sentiment_pairs": [
            {
                "topic": "Рефинансирование кредитов",
                "sentiment": "positive"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "positive"
            },
            {
                "topic": "Мобильное приложение",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 414832,
        "summarized_review": "Клиент положительно оценил современный и удобный процесс оформления автокредита. Для получения кредита понадобился только паспорт, а договор и карту с деньгами доставил курьер на дом.",
        "topic_sentiment_pairs": [
            {
                "topic": "Автокредиты",
                "sentiment": "positive"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "positive"
            }
        ]
    },
    {
        "id": 478166,
        "summarized_review": "Клиент возмущен тем, что сотрудники колл-центра отказываются озвучивать условия по предварительно одобренному кредиту по телефону, настойчиво приглашая в офис. Клиент расценивает это как манипулятивную тактику.",
        "topic_sentiment_pairs": [
            {
                "topic": "Потребительский кредит",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            }
        ]
    },
    {
        "id": 549954,
        "summarized_review": "Клиент очень доволен кредитной картой благодаря длинному льготному периоду и кэшбэку. Особенно отмечено высокое качество обслуживания: сотрудники все объяснили, быстро выдали карту и помогли с ее активацией в приложении.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредитные карты",
                "sentiment": "positive"
            },
            {
                "topic": "Офисное обслуживание",
                "sentiment": "positive"
            },
            {
                "topic": "Мобильное приложение",
                "sentiment": "positive"
            },
            {
                "topic": "Вклады",
                "sentiment": "neutral"
            }
        ]
    },
    {
        "id": 144538,
        "summarized_review": "У клиента были украдены деньги со счета. Он жалуется, что банк не расследует мошенничество, а вместо этого требует вернуть украденные средства с процентами.",
        "topic_sentiment_pairs": [
            {
                "topic": "Кредитные карты",
                "sentiment": "negative"
            },
            {
                "topic": "Дистанционное обслуживание",
                "sentiment": "negative"
            },
            {
                "topic": "Безопасность",
                "sentiment": "negative"
            }
        ]
    }
]

In [ ]:
# tested = pd.DataFrame(test)

In [18]:
df_randomised = df.sample(df.shape[0], random_state=42)

In [19]:
df_randomised

,review_id,date,review_text,topic,subtopic,sentiment
33,989095,2025-08-19,Пользовался дебетовой картой газпромбанка пару...,Дебетовые карты,NaN,Negative
555,904370,2024-12-02,Здравствуйте! Газпромбанк — самый худший банк!...,Вклады,NaN,Positive
3413,408232,2021-08-11,Обратилась в банк с просьбой о выдаче автокред...,Автокредиты,NaN,Negative
2420,547546,2022-07-13,"Хороший банк, вчера вечером позвонил, консульт...",Кредитные карты,«Удобная»,Positive
4424,299984,2018-10-09,Ошибка подключения телекард на телефон. При на...,Обслуживание,NaN,Negative
...,...,...,...,...,...,...
4426,299718,2018-10-06,"Просто днищенские карты, дали карту для получе...",Дебетовые карты,NaN,Negative
466,912227,2024-12-23,"Заказала дебетовую карту Газпромбанк, привез к...",Дебетовые карты,«Умная карта»,Positive
3092,463192,2021-11-17,"Скачала мобильный банкинг ГПБ, чтобы наконец р...",Дистанционное обслуживание,NaN,Positive
3772,393190,2021-01-26,"Сняли 99р тихо, не знаю за что, код операции 1...",Ипотека,NaN,Positive


In [12]:
df_start = 0
df_end = df_start + 400
df_start, df_end

(0, 400)

In [13]:
df_part_randomised = df_randomised.iloc[df_start:df_end].copy()

In [14]:
df_part_randomised["review_text"] = df_part_randomised["review_text"].str.replace("\n", " ")

In [15]:
texts = df_part_randomised.apply(lambda x: f"Отзыв {x['review_id']}: {x['review_text']}", axis=1)

In [16]:
combined_text = "\n\n".join(texts.values)

In [17]:
print(combined_text)

Отзыв 989095: Пользовался дебетовой картой газпромбанка пару лет и не ожидал такого подвоха. Списали 299 рублей за какую-то дополнительную услугу под названием газпром бонус. Я не давал согласия на доп. Услуги. Обратился в чат с банком. Мне ответили, что возврат не возможен. Хотя Госдума приняла закон о запрете списания средств банками за доп. Услуги без согласия пользователя, но банки продолжают это делать. Больше не буду пользоваться картой газпрома

Отзыв 904370: Здравствуйте! Газпромбанк — самый худший банк! Три дня стояла в очереди, чтобы получить готовую карту. Открыла накопительный счёт. Три недели лежали деньги, начисление процентов ежедневное. По окончании месяца я не получила ни копейки. Банк ответил, что на последний день месяца на счёте был нулевой остаток. Проценты не начисляются, за исключением месяца, в котором произведено первое пополнение счёта «. Ноябрь — первый месяц пополнения счёта. У меня деньги лежали 3 недели в Вашем банке! Это первый банк, который не выплатил п

In [175]:
with open(f'mount/data/gemma{df_start}-{df_end}.txt', 'w') as file:
    file.write(combined_text)

## Models

### Qwen-4B

In [5]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model_name = "Qwen/Qwen3-4B-Instruct-2507-FP8"

# # load the tokenizer and the model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     device_map="auto"
# )

# # prepare the model input
# prompt = "Give me a short introduction to large language model."
# messages = [
#     {"role": "user", "content": prompt}
# ]
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
# )
# model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# # conduct text completion
# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=16384
# )
# output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# content = tokenizer.decode(output_ids, skip_special_tokens=True)

# print("content:", content)

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=4096
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

content: A large language model (LLM) is a type of artificial intelligence model trained on vast amounts of text data to understand and generate human-like language. These models, such as GPT, BERT, and Llama, are designed to recognize patterns, context, and meaning in text, enabling them to answer questions, write stories, code, summarize information, and perform other language-related tasks. By using deep neural networks with billions of parameters, LLMs can generate coherent and contextually relevant responses. They are widely used in chatbots, content creation, translation, and more. While powerful, they can sometimes produce inaccurate or biased outputs, so critical evaluation and responsible use are essential.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto"},
    dtype="auto",
    device_map="auto"
)

In [ ]:
popular_topics = list(df["topic"].value_counts().iloc[:13].index)

popular_topics += ["Потребительский кредит", "Реструктаризация кредита"]

popular_topics

['Дебетовые карты',
 'Обслуживание',
 'Кредиты наличными',
 'Кредитные карты',
 'Вклады',
 'Дистанционное обслуживание',
 'Другие услуги',
 'Ипотека',
 'Автокредиты',
 'Рефинансирование кредитов',
 'Рефинансирование ипотеки',
 'Обмен валют',
 'Мобильное приложение',
 'Потребительский кредит',
 'Реструктаризация кредита']

In [24]:
print(df["review_text"].sample(1).values[0])

Заказав две дебетовые карты банка, пожалела, что связалась с этим банком. Карты получить не смогла, о чем предварительно (за день до доставки карт) позвонила на горячую линию и сообщила. Поддержка по телефону ответила спасибо, но предупредила, что скорее всего сотрулник не получит информацию об отказе. Очень интересно. Затем начались смс и звонки. От сотрудника доставки 3 звонка и несколько смс. И с этого момента, ежедневно, поступают звонки от Газпромбанка с формулировкой: Здравствуйте. Вы. ФИО. С учётом телефонного мошенничества, я не могу сообщить им запрошенные данные о чем и сообщаю. В ответ или: тогла мы перезвоним вам позже или опять повторяют запрос. В воскресенье поступило 5 таких звонков с разных номеров. Далее ещё 3 на следующий день. Перезвонив в банк, уточняю у сотрудника, являются ли все номера-номерами банка. Ответ — да. Оставляю заявление о запрете звонков и смс от банка. Но, запрет начинает работать через 72 часа после заявления. И тут же ещё
3 звонка. Объяснять что ли

In [28]:
reviews = list(df["review_text"].sample(5))

reviews = "\n\n".join([f"Отзыв {i+1}: {reviews[i]}" for i in range(len(reviews))])

добавить пример, имена сентиментов

In [52]:
system_prompt = f"""
Твоя задача - суммаризировать отзывы клиентов о банковских продуктах банка Газпромбанка (ГПБ). \
Для каждого отзыва тебе нужно вывести короткую версию отзыва, а также определить о каких продуктах/услугах говорит клиент. \
Примером таких продуктов/услуг может быть: {", ".join(popular_topics)}. Помимо этих могут возникать и другие.
Тебе также необходимо определить тональность клиента, по отношению к каждому определённому продукту, хорошо или плохо они отзываются о продукте. \
Тональность может быть одной из 3: positive, negative, neutral.

Для каждого отзыва ответь в следующем виде:
{{
[
    {{
        "summarized_review": "краткий текст первого отзыва",
        "products": ["продукт1", "продукт2"],
        "sentiments" : ["positive", "negative"]
    }},
    {{
        "summarized_review": "краткий текст второго отзыва",
        "products": ["продукт1", "продукт2", "продукт3"],
        "sentiments" : ["neutral", "neutral", "neutral"]
    }}
]
}}

Вот отзывы клиентов:
"""

messages = [
    [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": reviews
        },
    ],
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer(text, return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=4096
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

In [53]:
print(system_prompt, "\n-----------------------\n")

print(reviews, "\n-----------------------\n")

print(content)


Твоя задача - суммаризировать отзывы клиентов о банковских продуктах банка Газпромбанка (ГПБ). Для каждого отзыва тебе нужно вывести короткую версию отзыва, а также определить о каких продуктах/услугах говорит клиент. Примером таких продуктов/услуг может быть: Дебетовые карты, Обслуживание, Кредиты наличными, Кредитные карты, Вклады, Дистанционное обслуживание, Другие услуги, Ипотека, Автокредиты, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит, Реструктаризация кредита. Помимо этих могут возникать и другие.
Тебе также необходимо определить тональность клиента, по отношению к каждому определённому продукту, хорошо или плохо они отзываются о продукте. Тональность может быть одной из 3: positive, negative, neutral.

Для каждого отзыва ответь в следующем виде:
{
[
    {
        "summarized_review": "краткий текст первого отзыва",
        "products": ["продукт1", "продукт2"],
        "sentiments" : ["positive", "negative"]
   

### Qwen-0.6B

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# # quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# model_name = "Qwen/Qwen3-0.6B"

# # load the tokenizer and the model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     # quantization_config=quantization_config,
#     dtype="auto",
#     device_map="auto"
# )

# # prepare the model input
# prompt = "Give me a short introduction to large language model."
# messages = [
#     {"role": "user", "content": prompt}
# ]
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
# )
# model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# # conduct text completion
# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=4096
# )
# output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# content = tokenizer.decode(output_ids, skip_special_tokens=True)

# print("content:", content)


content: <think>
Okay, the user wants a short introduction to a large language model. Let me start by recalling what I know about them. Large language models are big language models, right? They use massive amounts of data to learn patterns and make decisions.

I should mention that they can understand and generate text in various languages. Maybe include something about their ability to handle different tasks like answering questions, writing articles, or even creating creative content. Also, it's important to note that they can't make up stories, which is a good point.

Wait, should I mention the data source? Like, they are trained on huge datasets, so that's a plus. Also, their performance can vary depending on the training data and the specific tasks. Maybe add something about how they can be used in various fields like healthcare, education, or business.

I need to keep it concise. Let me check if I'm missing any key points. The user wants a short introduction, so I should avoid g

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=quantization_config,
    attn_implementation = "flash_attention_2",
    dtype="auto",
    device_map="auto"
)

In [ ]:
# popular_topics = list(df["topic"].value_counts().iloc[:13].index)

# popular_topics += ["Потребительский кредит", "Реструктаризация кредита"]

# popular_topics

['Дебетовые карты',
 'Обслуживание',
 'Кредиты наличными',
 'Кредитные карты',
 'Вклады',
 'Дистанционное обслуживание',
 'Другие услуги',
 'Ипотека',
 'Автокредиты',
 'Рефинансирование кредитов',
 'Рефинансирование ипотеки',
 'Обмен валют',
 'Мобильное приложение',
 'Потребительский кредит',
 'Реструктаризация кредита']

In [ ]:
# reviews = list(df["review_text"].sample(5))

# reviews = "\n\n".join([f"Отзыв {i+1}: {reviews[i]}" for i in range(len(reviews))])

добавить пример, имена сентиментов, описание тем, уточнить что это темы и темы могут быть продуктами/услугами и тп

In [25]:
# system_prompt = f"""
# Твоя задача - суммаризировать отзывы клиентов о банковских продуктах банка Газпромбанка (ГПБ). \
# Для каждого отзыва тебе нужно вывести короткую версию отзыва, а также определить о каких продуктах/услугах говорит клиент. \
# Примером таких продуктов/услуг может быть: {", ".join(popular_topics)}. Помимо этих могут возникать и другие.
# Тебе также необходимо определить тональность клиента, по отношению к каждому определённому продукту, хорошо или плохо они отзываются о продукте. \
# Тональность может быть одной из 3: positive, negative, neutral.

# Для каждого отзыва ответь в следующем виде:
# {{
#     [
#         {{
#             "summarized_review": "краткий текст первого отзыва",
#             "products": ["продукт1", "продукт2"],
#             "sentiments" : ["positive", "negative"]
#         }},
#         {{
#             "summarized_review": "краткий текст второго отзыва",
#             "products": ["продукт1", "продукт2", "продукт3"],
#             "sentiments" : ["neutral", "neutral", "neutral"]
#         }}
#     ]
# }}

# Вот отзывы клиентов:
# """

# system_prompt = f"""
# Твоя задача - классифицировать отзывы клиентов о банковских продуктах банка Газпромбанка (ГПБ), \
# тебе нужно определить про какие продукты говорят клиенты и как они отзываются о этих продуктах.
# Для каждого отзыва тебе нужно определить о каких продуктах/услугах говорит клиент. \
# Примером таких продуктов/услуг может быть: {", ".join(popular_topics)}. Помимо этих могут возникать и другие.
# Тебе также необходимо определить тональность клиента, по отношению к каждому определённому продукту, хорошо или плохо они отзываются о продукте. \
# Тональность может быть одной из 3: positive, negative, neutral.

# Для каждого отзыва ответь в следующем виде:
# {{
# [
#     {{
#         "id" : 0,
#         "products": ["продукт1", "продукт2"],
#         "sentiments" : ["positive", "negative"]
#     }},
#     {{
#         "id" : 1,
#         "products": ["продукт1", "продукт2", "продукт3"],
#         "sentiments" : ["neutral", "neutral", "neutral"]
#     }}
# ]
# }}

# Вот отзывы клиентов:
# """

system_prompt = """\
Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Определить все продукты/услуги (темы), о которых упоминает клиент.
2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного упоминания продукта или услуги, не включай его.
-   **Символы `****`:** Это либо конфиденциальные данные (номера телефонов), либо ненормативная лексика. Учитывай общий контекст вокруг них для определения тональности.
-   **Если темы нет:** Если в отзыве невозможно определить ни одну тему, верни пустой массив `topic_sentiment_pairs`.

Вот список всех продуктов банка на сегодняшний день:
* **дебетовые карты:**
    * Умная дебетовая карта «Мир»
    * Премиальная карта Mir Supreme
    * Дебетовая карта с кэшбэком для самозанятых
    * Умная дебетовая карта «Мир»
    * Карта для автолюбителей «Газпромбанк—Газпромнефть»
    * Виртуальная дебетовая карта ГПБ&ФК «Зенит»
    * Дебетовая Пенсионная карта
* **Кредитные карты:**
    * Кредитная карта с льготным периодом до 120 дней
    * Простая кредитная карта
    * Кредитная карта 90 дней
    * Кредитная карта 180 дней Премиум
    * Кредитная карта для самозанятых
* **Накопительные счета:**
    * Накопительный счет
    * «Ежедневная выгода»
    * «Ежедневный процент»
    * «Премиум»
    * Социальный счет
* **Вклады:**
    * Вклад «Новые деньги»
    * Вклад «Ключевой момент»
    * Вклад «Копить»
    * Вклад «В Плюсе»
    * Вклад «Расширяй возможности»
    * Социальный вклад
* **Кредиты:**
    * Кредит наличными
    * Кредит наличными под залог недвижимости
    * Кредит на авто и другие цели
    * Рефинансирование потребительских кредитов
    * Дачный кредит
    * Кредит на образование
    * Кредит наличными для бюджетников
* **Другие услуги банка:**
    * Газпромбанк Мобайл
    * Газпромбанк Travel
    * Gazprom Pay
    * GorodPay
    * Газпром Бонус
    * Страховые и сервисные продукты
    * Депозитарные услуги


**Примеры:**
Отзыв 1: Мобильное приложение просто ужасное, постоянно вылетает. А вот вклад оформил недавно — условия хорошие.

Ответ:
{
    "id" : 1,
    "topic_sentiment_pairs": [
        {
            "topic": "Мобильное приложение",
            "sentiment": "negative"
        },
        {
            "topic": "Вклады",
            "sentiment": "positive"
        }
    ]
}


Отзыв 2: Звонил узнать насчет ипотеки, но в поддержке ничем не помогли.

Ответ:
{
    "id" : 2,
    "topic_sentiment_pairs": [
        {
            "topic": "Дистанционное обслуживание",
            "sentiment": "negative"
        },
        {
            "topic": "Ипотека",
            "sentiment": "neutral"
        }
    ]
}

**Вот отзывы клиентов, которые необходимо обработать:**
"""

after_prompt = "\nОтвет (только JSON, без дополнительного текста):"

review = df.sample(1)

review_text = review["review_text"].values[0]

review_text = "Отзыв 2903: " + review_text.replace('\n', ' ')

messages = [
    [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": review_text + after_prompt
        },
    ],
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer(text, return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=4096
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

In [ ]:
print(system_prompt, "\n-----------------------\n")

print(review_text + after_prompt, "\n-----------------------\n")

print(content)

Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Определить все продукты/услуги (темы), о которых упоминает клиент.
2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного уп

### Gemma

In [4]:
# from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")
# model = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m")

In [ ]:
# popular_topics = df["topic"].value_counts().iloc[:13].index

# popular_topics

Index(['Дебетовые карты', 'Обслуживание', 'Кредиты наличными',
       'Кредитные карты', 'Вклады', 'Дистанционное обслуживание',
       'Другие услуги', 'Ипотека', 'Автокредиты', 'Рефинансирование кредитов',
       'Рефинансирование ипотеки', 'Обмен валют', 'Мобильное приложение'],
      dtype='object', name='topic')

In [ ]:
# reviews = list(df["review_text"].sample(5))

In [ ]:
from transformers import pipeline
import torch

# pipe = pipeline(
#     "text-generation",
#     model="google/gemma-3-1b-it", 
#     device="cuda", 
#     dtype=torch.bfloat16,
#     temperature=0.5,
# )

# pipe = pipeline(
#     "text-generation",
#     model="google/gemma-3-270m-it", 
#     device="cuda", 
#     dtype=torch.bfloat16,
#     # temperature=0.1,
# )

pipe = pipeline(
    "text-generation",
    model="google/gemma-3-270m-it", 
    device="cuda", 
    dtype=torch.bfloat16,
    # temperature=0.1,
)


Device set to use cuda


In [38]:
system_prompt = """\
Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Определить все продукты/услуги (темы), о которых упоминает клиент.
2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного упоминания продукта или услуги, не включай его.
-   **Символы `****`:** Это либо конфиденциальные данные (номера телефонов), либо ненормативная лексика. Учитывай общий контекст вокруг них для определения тональности.
-   **Если темы нет:** Если в отзыве невозможно определить ни одну тему, верни пустой массив `topic_sentiment_pairs`.

Вот список всех продуктов банка на сегодняшний день:
* **дебетовые карты:**
    * Умная дебетовая карта «Мир»
    * Премиальная карта Mir Supreme
    * Дебетовая карта с кэшбэком для самозанятых
    * Умная дебетовая карта «Мир»
    * Карта для автолюбителей «Газпромбанк—Газпромнефть»
    * Виртуальная дебетовая карта ГПБ&ФК «Зенит»
    * Дебетовая Пенсионная карта
* **Кредитные карты:**
    * Кредитная карта с льготным периодом до 120 дней
    * Простая кредитная карта
    * Кредитная карта 90 дней
    * Кредитная карта 180 дней Премиум
    * Кредитная карта для самозанятых
* **Накопительные счета:**
    * Накопительный счет
    * «Ежедневная выгода»
    * «Ежедневный процент»
    * «Премиум»
    * Социальный счет
* **Вклады:**
    * Вклад «Новые деньги»
    * Вклад «Ключевой момент»
    * Вклад «Копить»
    * Вклад «В Плюсе»
    * Вклад «Расширяй возможности»
    * Социальный вклад
* **Кредиты:**
    * Кредит наличными
    * Кредит наличными под залог недвижимости
    * Кредит на авто и другие цели
    * Рефинансирование потребительских кредитов
    * Дачный кредит
    * Кредит на образование
    * Кредит наличными для бюджетников
* **Другие услуги банка:**
    * Газпромбанк Мобайл
    * Газпромбанк Travel
    * Gazprom Pay
    * GorodPay
    * Газпром Бонус
    * Страховые и сервисные продукты
    * Депозитарные услуги


**Примеры:**
Отзыв 1: Мобильное приложение просто ужасное, постоянно вылетает. А вот вклад оформил недавно — условия хорошие.

Ответ:
{
    "id" : 1,
    "topic_sentiment_pairs": [
        {
            "topic": "Мобильное приложение",
            "sentiment": "negative"
        },
        {
            "topic": "Вклады",
            "sentiment": "positive"
        }
    ]
}


Отзыв 2: Звонил узнать насчет ипотеки, но в поддержке ничем не помогли.

Ответ:
{
    "id" : 2,
    "topic_sentiment_pairs": [
        {
            "topic": "Дистанционное обслуживание",
            "sentiment": "negative"
        },
        {
            "topic": "Ипотека",
            "sentiment": "neutral"
        }
    ]
}

**Вот отзывы клиентов, которые необходимо обработать:**
"""

after_prompt = "\nОтвет (только JSON, без дополнительного текста):"

review = df.sample(1)

review_text = review["review_text"].values[0]

review_text = "Отзыв 2903: " + review_text.replace('\n', ' ')

messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_prompt},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": review_text + after_prompt},]
        },
    ],
]

output = pipe(messages, max_new_tokens=256)

In [39]:
print(system_prompt, "\n-----------------------\n")

print(review_text + after_prompt, "\n-----------------------\n")

print(output[0][0]["generated_text"][-1]["content"])

Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Определить все продукты/услуги (темы), о которых упоминает клиент.
2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного уп